In [ ]:
import os
import gc
import math
import copy
import random
import warnings
from collections import Counter

import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    accuracy_score
)

warnings.filterwarnings("ignore")

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cuda


In [ ]:
import kagglehub

In [ ]:
path = kagglehub.dataset_download(
    "ayenuryrr/loghub-hdfs-hadoop-distributed-file-system-data"
)
print("Dataset path:", path)

base_path = path + "/HDFS_v1/preprocessed"
traces_path = base_path + "/Event_traces.csv"

trace_df = pd.read_csv(traces_path)
print("trace_df shape:", trace_df.shape)
print(trace_df.head())

Dataset path: C:\Users\serge\.cache\kagglehub\datasets\ayenuryrr\loghub-hdfs-hadoop-distributed-file-system-data\versions\3
trace_df shape: (575061, 6)
                    BlockId    Label  Type  \
0  blk_-1608999687919862906  Success   NaN   
1   blk_7503483334202473044  Success   NaN   
2  blk_-3544583377289625738     Fail  21.0   
3  blk_-9073992586687739851  Success   NaN   
4   blk_7854771516489510256  Success   NaN   

                                            Features  \
0  [E5,E22,E5,E5,E11,E11,E9,E9,E11,E9,E26,E26,E26...   
1  [E5,E5,E22,E5,E11,E9,E11,E9,E11,E9,E26,E26,E26...   
2  [E5,E22,E5,E5,E11,E9,E11,E9,E11,E9,E3,E26,E26,...   
3  [E5,E22,E5,E5,E11,E9,E11,E9,E11,E9,E26,E26,E26...   
4  [E5,E5,E22,E5,E11,E9,E11,E9,E11,E9,E26,E26,E26...   

                                        TimeInterval  Latency  
0  [0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...     3802  
1  [0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...     3802  
2  [0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0

In [ ]:
def parse_sequence(s):
    if pd.isna(s):
        return []

    s = str(s).strip()

    if s == "":
        return []

    s = s.strip("[]")

    if s == "":
        return []

    items = [item.strip().strip("'").strip('"') for item in s.split(",")]
    items = [item for item in items if item != ""]

    return items

def parse_float_sequence(s):
    if pd.isna(s):
        return []

    s = str(s).strip()

    if s == "":
        return []

    s = s.strip("[]")

    if s == "":
        return []

    result = []
    for item in s.split(","):
        item = item.strip()
        if item == "":
            continue
        try:
            result.append(float(item))
        except ValueError:
            continue

    return result

In [ ]:
df = trace_df.copy()

df["sequence"] = df["Features"].apply(parse_sequence)

df["label"] = df["Label"].map({
    "Success": 0,
    "Fail": 1
})

df["seq_len"] = df["sequence"].apply(len)

print("\nInitial stats")
print("Original size:", len(df))
print("Missing labels:", df["label"].isna().sum())
print("Empty sequences:", (df["seq_len"] == 0).sum())
print(df["label"].value_counts(dropna=False))
print(df["seq_len"].describe())


df = df[df["label"].notna()].copy()
df = df[df["seq_len"] > 0].copy()


MIN_SEQ_LEN = 5
df = df[df["seq_len"] >= MIN_SEQ_LEN].copy()


print("\nAfter basic cleaning:", len(df))
print(df["label"].value_counts())


Initial stats
Original size: 575061
Missing labels: 0
Empty sequences: 0
label
0    558223
1     16838
Name: count, dtype: int64
count    575061.000000
mean         19.433815
std           5.177735
min           2.000000
25%          19.000000
50%          19.000000
75%          20.000000
max         298.000000
Name: seq_len, dtype: float64

After basic cleaning: 568880
label
0    558223
1     10657
Name: count, dtype: int64


In [ ]:

feature_label_counts = df.groupby("Features")["label"].nunique()
conflicting_features = feature_label_counts[feature_label_counts > 1].index

print("\nConflicting sequences:", len(conflicting_features))

df_clean = df[~df["Features"].isin(conflicting_features)].copy()

print("After removing conflicts:", len(df_clean))
print(df_clean["label"].value_counts())
print("Unique feature patterns:", df_clean["Features"].nunique())


Conflicting sequences: 10
After removing conflicts: 568834
label
0    558201
1     10633
Name: count, dtype: int64
Unique feature patterns: 18356


In [ ]:

feature_labels = (
    df_clean.groupby("Features")["label"]
    .agg(lambda x: x.mode().iloc[0])
    .reset_index()
)

print("\nfeature_labels shape:", feature_labels.shape)
print(feature_labels["label"].value_counts())

train_feat, temp_feat = train_test_split(
    feature_labels,
    test_size=0.30,
    stratify=feature_labels["label"],
    random_state=42
)

val_feat, test_feat = train_test_split(
    temp_feat,
    test_size=0.50,
    stratify=temp_feat["label"],
    random_state=42
)

train_features = set(train_feat["Features"])
val_features = set(val_feat["Features"])
test_features = set(test_feat["Features"])

train_df = df_clean[df_clean["Features"].isin(train_features)].copy()
val_df = df_clean[df_clean["Features"].isin(val_features)].copy()
test_df = df_clean[df_clean["Features"].isin(test_features)].copy()


def summarize_split(name, split_df):
    print(f"\n{name}")
    print("  instances:", len(split_df))
    print("  unique patterns:", split_df["Features"].nunique())
    print("  label counts:")
    print(split_df["label"].value_counts())
    print("  anomaly ratio:", round(split_df["label"].mean(), 4))


summarize_split("Train", train_df)
summarize_split("Validation", val_df)
summarize_split("Test", test_df)

train_set = set(train_df["Features"].astype(str))
val_set = set(val_df["Features"].astype(str))
test_set = set(test_df["Features"].astype(str))

print("\nOverlap check")
print("Train ∩ Val :", len(train_set & val_set))
print("Train ∩ Test:", len(train_set & test_set))
print("Val ∩ Test  :", len(val_set & test_set))

train_seq_set = set(train_df["sequence"].apply(tuple))
val_seq_set = set(val_df["sequence"].apply(tuple))
test_seq_set = set(test_df["sequence"].apply(tuple))

print("\nSequence overlap check")
print("Train ∩ Val :", len(train_seq_set & val_seq_set))
print("Train ∩ Test:", len(train_seq_set & test_seq_set))
print("Val ∩ Test  :", len(val_seq_set & test_seq_set))


feature_labels shape: (18356, 2)
label
0    14249
1     4107
Name: count, dtype: int64

Train
  instances: 359655
  unique patterns: 12849
  label counts:
label
0    352608
1      7047
Name: count, dtype: int64
  anomaly ratio: 0.0196

Validation
  instances: 71974
  unique patterns: 2753
  label counts:
label
0    70237
1     1737
Name: count, dtype: int64
  anomaly ratio: 0.0241

Test
  instances: 137205
  unique patterns: 2754
  label counts:
label
0    135356
1      1849
Name: count, dtype: int64
  anomaly ratio: 0.0135

Overlap check
Train ∩ Val : 0
Train ∩ Test: 0
Val ∩ Test  : 0

Sequence overlap check
Train ∩ Val : 0
Train ∩ Test: 0
Val ∩ Test  : 0


In [ ]:

normal_train_df = train_df[train_df["label"] == 0].copy()

print("\nNormal train size:", len(normal_train_df))
print("Normal train unique patterns:", normal_train_df["Features"].nunique())


Normal train size: 352608
Normal train unique patterns: 9974


In [ ]:

from collections import Counter

def build_vocab(sequences, min_freq=1):
    counter = Counter()
    for seq in sequences:
        counter.update(seq)

    vocab = {
        "<PAD>": 0,
        "<UNK>": 1
    }

    for token, count in counter.items():
        if count >= min_freq:
            vocab[token] = len(vocab)

    return vocab, counter

vocab, token_counter = build_vocab(normal_train_df["sequence"], min_freq=1)

print("\nVocab size:", len(vocab))
print("Top 10 tokens:", token_counter.most_common(10))


Vocab size: 18
Top 10 tokens: [('E26', 1060890), ('E5', 1059497), ('E11', 1057824), ('E9', 1057824), ('E21', 899191), ('E23', 897718), ('E22', 352608), ('E3', 308292), ('E4', 247096), ('E2', 83494)]


In [ ]:

def encode_sequence(seq, vocab):
    return [vocab.get(token, vocab["<UNK>"]) for token in seq]

for split_name, split_df in [
    ("train", train_df),
    ("val", val_df),
    ("test", test_df),
    ("normal_train", normal_train_df),
]:
    split_df["sequence_ids"] = split_df["sequence"].apply(lambda x: encode_sequence(x, vocab))
    split_df["seq_len_ids"] = split_df["sequence_ids"].apply(len)
    print(f"\n{split_name} encoded")
    print(split_df["seq_len_ids"].describe())



MAX_LEN = int(np.percentile(normal_train_df["seq_len_ids"], 95))
MAX_LEN = max(MAX_LEN, 5)

print("\nChosen MAX_LEN:", MAX_LEN)


train encoded
count    359655.000000
mean         20.048299
std           4.889883
min           8.000000
25%          19.000000
50%          19.000000
75%          22.000000
max         280.000000
Name: seq_len_ids, dtype: float64

val encoded
count    71974.000000
mean        17.545767
std          5.995525
min         13.000000
25%         13.000000
50%         14.000000
75%         20.000000
max        278.000000
Name: seq_len_ids, dtype: float64

test encoded
count    137205.000000
mean         19.547648
std           3.969623
min           8.000000
25%          19.000000
50%          19.000000
75%          19.000000
max         298.000000
Name: seq_len_ids, dtype: float64

normal_train encoded
count    352608.000000
mean         19.940347
std           4.765715
min          13.000000
25%          19.000000
50%          19.000000
75%          22.000000
max         280.000000
Name: seq_len_ids, dtype: float64

Chosen MAX_LEN: 28


In [ ]:
def pad_or_truncate_prefix(seq, max_len, pad_value=0):
    if len(seq) > max_len:
        seq = seq[-max_len:]
    return seq + [pad_value] * (max_len - len(seq))


import torch
from torch.utils.data import Dataset

class NextEventDataset(Dataset):
    def __init__(self, dataframe, max_len):
        self.samples = []
        self.max_len = max_len

        for _, row in dataframe.iterrows():
            seq = row["sequence_ids"]

            if len(seq) < 2:
                continue

            for i in range(1, len(seq)):
                prefix = seq[:i]
                target = seq[i]

                if len(prefix) > self.max_len:
                    prefix = prefix[-self.max_len:]

                length = len(prefix)

                prefix_padded = prefix + [0] * (self.max_len - length)
                attention_mask = [1] * length + [0] * (self.max_len - length)

                self.samples.append({
                    "input_ids": prefix_padded,
                    "attention_mask": attention_mask,
                    "length": length,
                    "target": target
                })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        return {
            "input_ids": torch.tensor(item["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(item["attention_mask"], dtype=torch.long),
            "length": torch.tensor(item["length"], dtype=torch.long),
            "target": torch.tensor(item["target"], dtype=torch.long)
        }

In [ ]:
normal_val_df = val_df[val_df["label"] == 0].copy()

train_next_dataset = NextEventDataset(normal_train_df, max_len=MAX_LEN)
normal_val_next_dataset = NextEventDataset(normal_val_df, max_len=MAX_LEN)
test_next_dataset = NextEventDataset(test_df, max_len=MAX_LEN)

print("Train next-event samples:", len(train_next_dataset))
print("Normal val next-event samples:", len(normal_val_next_da taset))
print("Test next-event samples:", len(test_next_dataset))

Train next-event samples: 6678518
Normal val next-event samples: 1149936
Test next-event samples: 2544830


In [ ]:
from torch.utils.data import DataLoader

In [ ]:
BATCH_SIZE = 256

train_loader = DataLoader(train_next_dataset, batch_size=BATCH_SIZE, shuffle=True)
normal_val_loader = DataLoader(normal_val_next_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_next_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
del trace_df
gc.collect()

0

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)  # [1, max_len, d_model]
        self.register_buffer("pe", pe)

    def forward(self, x):
        seq_len = x.size(1)
        return x + self.pe[:, :seq_len]

In [ ]:
class TransformerNextEvent(nn.Module):
    def __init__(
        self,
        vocab_size,
        d_model=128,
        nhead=4,
        num_layers=2,
        dim_feedforward=256,
        dropout=0.2,
        max_len=128,
        pad_id=0
    ):
        super().__init__()

        self.pad_id = pad_id
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.pos_encoding = PositionalEncoding(d_model, max_len=max_len)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(d_model, vocab_size)

    def forward(self, input_ids, attention_mask):


        x = self.embedding(input_ids)
        x = self.pos_encoding(x)

        key_padding_mask = (attention_mask == 0)

        x = self.encoder(
            x,
            src_key_padding_mask=key_padding_mask
        )

        lengths = attention_mask.sum(dim=1) - 1
        lengths = torch.clamp(lengths, min=0)

        batch_idx = torch.arange(x.size(0), device=x.device)
        last_hidden = x[batch_idx, lengths]

        logits = self.fc(self.dropout(last_hidden))
        return logits

In [ ]:
model = TransformerNextEvent(
    vocab_size=len(vocab),
    d_model=128,
    nhead=4,
    num_layers=2,
    dim_feedforward=256,
    dropout=0.2,
    max_len=MAX_LEN,
    pad_id=vocab["<PAD>"]
).to(device)

print(model)
print("Trainable params:", sum(p.numel() for p in model.parameters() if p.requires_grad))

TransformerNextEvent(
  (embedding): Embedding(18, 128, padding_idx=0)
  (pos_encoding): PositionalEncoding()
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear(in_features=128, out_features=256, bias=True)
        (dropout): Dropout(p=0.2, inplace=False)
        (linear2): Linear(in_features=256, out_features=128, bias=True)
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.2, inplace=False)
        (dropout2): Dropout(p=0.2, inplace=False)
      )
    )
  )
  (dropout): Dropout(p=0.2, inplace=False)
  (fc): Linear(in_features=128, out_features=18, bias=True)
)
Trainable params: 269586


In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device, max_grad_norm=1.0):
    model.train()
    total_loss = 0.0

    for batch in tqdm(loader, desc="train", leave=False):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        targets = batch["target"].to(device)

        optimizer.zero_grad()

        logits = model(input_ids, attention_mask)
        loss = criterion(logits, targets)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [ ]:
@torch.no_grad()
def evaluate_next_event(model, loader, criterion, device):
    model.eval()

    total_loss = 0.0
    all_targets = []
    all_preds = []

    for batch in tqdm(loader, desc="eval", leave=False):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        targets = batch["target"].to(device)

        logits = model(input_ids, attention_mask)
        loss = criterion(logits, targets)

        preds = torch.argmax(logits, dim=1)

        total_loss += loss.item()
        all_targets.extend(targets.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())

    return {
        "loss": total_loss / len(loader),
        "accuracy": accuracy_score(all_targets, all_preds)
    }

In [ ]:
def fit_transformer_model(config, train_loader, val_loader, vocab_size, device, pad_id=0):
    model = TransformerNextEvent(
        vocab_size=vocab_size,
        d_model=config["d_model"],
        nhead=config["nhead"],
        num_layers=config["num_layers"],
        dim_feedforward=config["dim_feedforward"],
        dropout=config["dropout"],
        max_len=config["max_len"],
        pad_id=pad_id
    ).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=config["lr"],
        weight_decay=config["weight_decay"]
    )

    best_state = None
    best_val_loss = float("inf")
    best_epoch = -1
    patience_counter = 0
    history = []

    for epoch in range(config["num_epochs"]):
        train_loss = train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            criterion=criterion,
            device=device,
            max_grad_norm=config.get("max_grad_norm", 1.0)
        )

        val_metrics = evaluate_next_event(
            model=model,
            loader=val_loader,
            criterion=criterion,
            device=device
        )

        history.append({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_metrics["loss"],
            "val_accuracy": val_metrics["accuracy"]
        })

        print(
            f"Epoch {epoch+1}/{config['num_epochs']} | "
            f"train_loss={train_loss:.4f} | "
            f"val_loss={val_metrics['loss']:.4f} | "
            f"val_acc={val_metrics['accuracy']:.4f}"
        )

        if val_metrics["loss"] < best_val_loss:
            best_val_loss = val_metrics["loss"]
            best_epoch = epoch + 1
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= config["patience"]:
            print("Early stopping")
            break

    model.load_state_dict(best_state)

    return {
        "model": model,
        "history": pd.DataFrame(history),
        "best_val_loss": best_val_loss,
        "best_epoch": best_epoch,
        "num_params": count_parameters(model)
    }

In [ ]:
baseline_config = {
    "d_model": 128,
    "nhead": 4,
    "num_layers": 2,
    "dim_feedforward": 256,
    "dropout": 0.2,
    "lr": 3e-4,
    "weight_decay": 1e-4,
    "max_len": MAX_LEN,
    "num_epochs": 10,
    "patience": 3,
    "max_grad_norm": 1.0
}

In [ ]:
fit_result = fit_transformer_model(
    config=baseline_config,
    train_loader=train_loader,
    val_loader=normal_val_loader,
    vocab_size=len(vocab),
    device=device,
    pad_id=vocab["<PAD>"]
)

baseline_model = fit_result["model"]
history_df = fit_result["history"]

print("Best epoch:", fit_result["best_epoch"])
print("Best val loss:", fit_result["best_val_loss"])
print("Num params:", fit_result["num_params"])

history_df

Epoch 1/10 | train_loss=0.2416 | val_loss=0.3034 | val_acc=0.8732


Epoch 2/10 | train_loss=0.2246 | val_loss=0.3097 | val_acc=0.8763


Epoch 3/10 | train_loss=0.2225 | val_loss=0.3217 | val_acc=0.8713


Epoch 4/10 | train_loss=0.2212 | val_loss=0.3112 | val_acc=0.8743
Early stopping
Best epoch: 1
Best val loss: 0.3033533270536314
Num params: 269586


,epoch,train_loss,val_loss,val_accuracy
0,1,0.241630,0.303353,0.873167
1,2,0.224570,0.309732,0.876257
2,3,0.222546,0.321653,0.871253
3,4,0.221198,0.311238,0.874298


In [ ]:
def evaluate_anomaly_scores(test_labels, test_scores, threshold):
    preds = (np.array(test_scores) >= threshold).astype(int)
    y_true = np.array(test_labels)

    return {
        "precision": precision_score(y_true, preds, zero_division=0),
        "recall": recall_score(y_true, preds, zero_division=0),
        "f1": f1_score(y_true, preds, zero_division=0),
        "roc_auc": roc_auc_score(y_true, test_scores),
        "pr_auc": average_precision_score(y_true, test_scores)
    }

In [ ]:
def find_best_threshold(scores, labels):
    scores = np.array(scores)
    labels = np.array(labels)

    thresholds = np.unique(scores)

    best = {
        "threshold": None,
        "f1": -1,
        "precision": None,
        "recall": None
    }

    for thr in thresholds:
        preds = (scores >= thr).astype(int)
        f1 = f1_score(labels, preds, zero_division=0)

        if f1 > best["f1"]:
            best["threshold"] = thr
            best["f1"] = f1
            best["precision"] = precision_score(labels, preds, zero_division=0)
            best["recall"] = recall_score(labels, preds, zero_division=0)

    return best

In [ ]:
@torch.no_grad()
def transformer_anomaly_score(seq, model, vocab, max_len, device, agg="mean", top_k=3):
    model.eval()

    seq_ids = [vocab.get(token, vocab["<UNK>"]) for token in seq]
    step_scores = []

    for i in range(1, len(seq_ids)):
        prefix = seq_ids[:i]
        target = seq_ids[i]

        if len(prefix) > max_len:
            prefix = prefix[-max_len:]

        length = len(prefix)
        prefix_padded = prefix + [vocab["<PAD>"]] * (max_len - length)
        attention_mask = [1] * length + [0] * (max_len - length)

        input_ids = torch.tensor([prefix_padded], dtype=torch.long, device=device)
        attention_mask = torch.tensor([attention_mask], dtype=torch.long, device=device)

        logits = model(input_ids, attention_mask)
        probs = torch.softmax(logits, dim=1)

        prob = probs[0, target].item()
        step_scores.append(-np.log(prob + 1e-12))

    if len(step_scores) == 0:
        return 0.0

    step_scores = np.array(step_scores, dtype=np.float32)

    if agg == "mean":
        return float(np.mean(step_scores))
    elif agg == "max":
        return float(np.max(step_scores))
    elif agg == "topk_mean":
        k = min(top_k, len(step_scores))
        return float(np.mean(np.sort(step_scores)[-k:]))
    elif agg == "p95":
        return float(np.percentile(step_scores, 95))
    else:
        raise ValueError(f"Unknown agg: {agg}")

In [ ]:
def evaluate_transformer_detector(model, val_df, test_df, vocab, max_len, device, agg="mean", top_k=3):
    val_scores = val_df["sequence"].apply(
        lambda x: transformer_anomaly_score(
            seq=x,
            model=model,
            vocab=vocab,
            max_len=max_len,
            device=device,
            agg=agg,
            top_k=top_k
        )
    )

    test_scores = test_df["sequence"].apply(
        lambda x: transformer_anomaly_score(
            seq=x,
            model=model,
            vocab=vocab,
            max_len=max_len,
            device=device,
            agg=agg,
            top_k=top_k
        )
    )

    best_thr = find_best_threshold(val_scores, val_df["label"])
    test_metrics = evaluate_anomaly_scores(test_df["label"], test_scores, best_thr["threshold"])

    return {
        "threshold": best_thr["threshold"],
        "val_f1": best_thr["f1"],
        "val_precision": best_thr["precision"],
        "val_recall": best_thr["recall"],
        **test_metrics
    }

In [ ]:
baseline_metrics = evaluate_transformer_detector(
    model=baseline_model,
    val_df=val_df,
    test_df=test_df,
    vocab=vocab,
    max_len=baseline_config["max_len"],
    device=device,
    agg="mean",
    top_k=3
)

baseline_metrics

{'threshold': np.float64(0.7960872054100037),
 'val_f1': 0.6356968215158925,
 'val_precision': 0.6775244299674267,
 'val_recall': 0.5987334484743811,
 'precision': 0.7202495201535508,
 'recall': 0.811790156841536,
 'f1': 0.7632850241545893,
 'roc_auc': 0.9969153154861412,
 'pr_auc': 0.8009795299540243}

In [ ]:
tuning_configs = [
    {
        "run_name": "dropout_01",
        "d_model": 128,
        "nhead": 4,
        "num_layers": 2,
        "dim_feedforward": 256,
        "dropout": 0.1,
        "lr": 3e-4,
        "weight_decay": 1e-4,
        "max_len": MAX_LEN,
        "num_epochs": 7,
        "patience": 3,
        "max_grad_norm": 1.0
    },
    {
        "run_name": "dropout_03",
        "d_model": 128,
        "nhead": 4,
        "num_layers": 2,
        "dim_feedforward": 256,
        "dropout": 0.3,
        "lr": 3e-4,
        "weight_decay": 1e-4,
        "max_len": MAX_LEN,
        "num_epochs": 7,
        "patience": 3,
        "max_grad_norm": 1.0
    },
    {
        "run_name": "lr_1e3",
        "d_model": 128,
        "nhead": 4,
        "num_layers": 2,
        "dim_feedforward": 256,
        "dropout": 0.2,
        "lr": 1e-3,
        "weight_decay": 1e-4,
        "max_len": MAX_LEN,
        "num_epochs": 7,
        "patience": 3,
        "max_grad_norm": 1.0
    },
    {
        "run_name": "dmodel_64",
        "d_model": 64,
        "nhead": 4,
        "num_layers": 2,
        "dim_feedforward": 128,
        "dropout": 0.2,
        "lr": 3e-4,
        "weight_decay": 1e-4,
        "max_len": MAX_LEN,
        "num_epochs": 7,
        "patience": 3,
        "max_grad_norm": 1.0
    },
    {
        "run_name": "dmodel_256",
        "d_model": 256,
        "nhead": 8,
        "num_layers": 2,
        "dim_feedforward": 512,
        "dropout": 0.2,
        "lr": 3e-4,
        "weight_decay": 1e-4,
        "max_len": MAX_LEN,
        "num_epochs": 7,
        "patience": 3,
        "max_grad_norm": 1.0
    },
    {
        "run_name": "layers_3",
        "d_model": 128,
        "nhead": 4,
        "num_layers": 3,
        "dim_feedforward": 256,
        "dropout": 0.2,
        "lr": 3e-4,
        "weight_decay": 1e-4,
        "max_len": MAX_LEN,
        "num_epochs": 7,
        "patience": 3,
        "max_grad_norm": 1.0
    },
    {
        "run_name": "maxlen_50",
        "d_model": 128,
        "nhead": 4,
        "num_layers": 2,
        "dim_feedforward": 256,
        "dropout": 0.2,
        "lr": 3e-4,
        "weight_decay": 1e-4,
        "max_len": 50,
        "num_epochs": 7,
        "patience": 3,
        "max_grad_norm": 1.0
    }
]

In [ ]:
import numpy as np

def score_sequences_transformer(model, df, vocab, max_len, device, agg="mean", top_k=3):
    scores = []

    model.eval()

    for seq in df["sequence"].values:

        score = transformer_anomaly_score(
            seq=seq,
            model=model,
            vocab=vocab,
            max_len=max_len,
            device=device,
            agg=agg,
            top_k=top_k
        )

        scores.append(score)

    return np.array(scores)

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score, average_precision_score, roc_auc_score

def evaluate_transformer_on_val_only(
    model,
    val_df,
    vocab,
    max_len,
    device,
    agg="mean",
    top_k=3
):
    val_scores = score_sequences_transformer(
        model=model,
        df=val_df,
        vocab=vocab,
        max_len=max_len,
        device=device,
        agg=agg,
        top_k=top_k
    )

    val_labels = val_df["label"].values
    best_result=find_best_threshold(val_scores, val_labels)
    best_thr = best_result['threshold']
    best_val_f1 = best_result['f1']
    val_pred = (val_scores >= best_thr).astype(int)

    result = {
        "val_f1": f1_score(val_labels, val_pred, zero_division=0),
        "val_precision": precision_score(val_labels, val_pred, zero_division=0),
        "val_recall": recall_score(val_labels, val_pred, zero_division=0),
        "val_pr_auc": average_precision_score(val_labels, val_scores),
        "val_roc_auc": roc_auc_score(val_labels, val_scores),
        "threshold": best_thr
    }

    return result

In [ ]:
val_tune_df = val_df.groupby("label", group_keys=False).apply(
    lambda x: x.sample(n=min(1000, len(x)), random_state=42)
).reset_index(drop=True)

print("Full val size:", len(val_df))
print("Tune val size:", len(val_tune_df))
print(val_tune_df["label"].value_counts())

Full val size: 71974
Tune val size: 2000
label
0    1000
1    1000
Name: count, dtype: int64


In [ ]:
import gc
import time
import copy
import torch
import pandas as pd
from torch.utils.data import DataLoader

tuning_results = []

best_model_state = None
best_model_config = None
best_model_run_name = None
best_score = -1

print(f"Total configs: {len(tuning_configs)}")

for i, config in enumerate(tuning_configs, 1):
    print("\n" + "=" * 80)
    print(f"Config {i}/{len(tuning_configs)} | {config['run_name']}")
    print(config)
    print("=" * 80)

    start_time = time.time()

    current_max_len = config["max_len"]
    current_batch_size = config.get("batch_size", BATCH_SIZE)

    print("Building datasets...")
    train_next_dataset = NextEventDataset(normal_train_df, max_len=current_max_len)
    normal_val_next_dataset = NextEventDataset(normal_val_df, max_len=current_max_len)

    print("Building dataloaders...")
    train_loader = DataLoader(
        train_next_dataset,
        batch_size=current_batch_size,
        shuffle=True
    )
    normal_val_loader = DataLoader(
        normal_val_next_dataset,
        batch_size=current_batch_size,
        shuffle=False
    )

    print("Training started...")
    fit_result = fit_transformer_model(
        config=config,
        train_loader=train_loader,
        val_loader=normal_val_loader,
        vocab_size=len(vocab),
        device=device,
        pad_id=vocab["<PAD>"]
    )
    print("Training finished.")

    current_model = fit_result["model"]

    print("Validation-only evaluation started...")
    val_result = evaluate_transformer_on_val_only(
        model=current_model,
        val_df=val_tune_df,
        vocab=vocab,
        max_len=current_max_len,
        device=device,
        agg="mean",
        top_k=3
    )
    print("Validation-only evaluation finished.")

    val_f1 = val_result["val_f1"]

    tuning_results.append({
        "run_name": config["run_name"],
        "d_model": config["d_model"],
        "nhead": config["nhead"],
        "num_layers": config["num_layers"],
        "dim_feedforward": config["dim_feedforward"],
        "dropout": config["dropout"],
        "lr": config["lr"],
        "weight_decay": config["weight_decay"],
        "max_len": config["max_len"],
        "num_epochs": config["num_epochs"],
        "patience": config["patience"],
        "val_f1": val_result["val_f1"],
        "val_precision": val_result["val_precision"],
        "val_recall": val_result["val_recall"],
        "val_pr_auc": val_result["val_pr_auc"],
        "val_roc_auc": val_result["val_roc_auc"],
        "threshold": val_result["threshold"],
        "elapsed_sec": round(time.time() - start_time, 2)
    })

    print(
        f"Done: {config['run_name']} | "
        f"val_f1={val_result['val_f1']:.4f} | "
        f"val_pr_auc={val_result['val_pr_auc']:.4f} | "
        f"time={round(time.time() - start_time, 2)} sec"
    )

    if val_f1 > best_score:
        best_score = val_f1
        best_model_state = copy.deepcopy(current_model.state_dict())
        best_model_config = copy.deepcopy(config)
        best_model_run_name = config["run_name"]
        print(f"🔥 New BEST model: {best_model_run_name} (val_f1={best_score:.4f})")

    del current_model
    del train_next_dataset, normal_val_next_dataset, train_loader, normal_val_loader, fit_result
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n" + "=" * 80)
print("TUNING FINISHED")
print(f"Best run: {best_model_run_name}")
print(f"Best val_f1: {best_score:.4f}")

tuning_results_df = pd.DataFrame(tuning_results).sort_values(
    by=["val_f1", "val_pr_auc"],
    ascending=False
).reset_index(drop=True)

tuning_results_df = tuning_results_df.round(4)
display(tuning_results_df)

Total configs: 7

Config 1/7 | dropout_01
{'run_name': 'dropout_01', 'd_model': 128, 'nhead': 4, 'num_layers': 2, 'dim_feedforward': 256, 'dropout': 0.1, 'lr': 0.0003, 'weight_decay': 0.0001, 'max_len': 28, 'num_epochs': 7, 'patience': 3, 'max_grad_norm': 1.0}
Building datasets...
Building dataloaders...
Training started...


Epoch 1/7 | train_loss=0.2326 | val_loss=0.3023 | val_acc=0.8742


Epoch 2/7 | train_loss=0.2195 | val_loss=0.3009 | val_acc=0.8771


Epoch 3/7 | train_loss=0.2177 | val_loss=0.3096 | val_acc=0.8749


Epoch 4/7 | train_loss=0.2167 | val_loss=0.3203 | val_acc=0.8604


...

Epoch 3/7 | train_loss=0.2227 | val_loss=0.3068 | val_acc=0.8743


Epoch 4/7 | train_loss=0.2214 | val_loss=0.3103 | val_acc=0.8764


Epoch 5/7 | train_loss=0.2206 | val_loss=0.3124 | val_acc=0.8747


Epoch 6/7 | train_loss=0.2200 | val_loss=0.3094 | val_acc=0.8765
Early stopping
Training finished.
Validation-only evaluation started...
Validation-only evaluation finished.
Done: maxlen_50 | val_f1=0.9803 | val_pr_auc=0.9897 | time=3598.09 sec

TUNING FINISHED
Best run: dropout_01
Best val_f1: 0.9841


,run_name,d_model,nhead,num_layers,dim_feedforward,dropout,lr,weight_decay,max_len,num_epochs,patience,val_f1,val_precision,val_recall,val_pr_auc,val_roc_auc,threshold,elapsed_sec
0,dropout_01,128,4,2,256,0.1,0.0003,0.0001,28,7,3,0.9841,0.9764,0.992,0.9924,0.9936,0.5648,2487.14
1,layers_3,128,4,3,256,0.2,0.0003,0.0001,28,7,3,0.9818,0.9661,0.998,0.9874,0.9901,0.5257,2351.29
2,dropout_03,128,4,2,256,0.3,0.0003,0.0001,28,7,3,0.9811,0.9781,0.984,0.9907,0.9923,0.6315,2451.22
3,dmodel_64,64,4,2,128,0.2,0.0003,0.0001,28,7,3,0.9807,0.9715,0.990,0.9904,0.9920,0.5514,3141.70
4,maxlen_50,128,4,2,256,0.2,0.0003,0.0001,50,7,3,0.9803,0.9642,0.997,0.9897,0.9916,0.5320,3598.09
5,lr_1e3,128,4,2,256,0.2,0.0010,0.0001,28,7,3,0.9782,0.9714,0.985,0.9856,0.9893,0.5666,2008.15
6,dmodel_256,256,8,2,512,0.2,0.0003,0.0001,28,7,3,0.9757,0.9676,0.984,0.9864,0.9892,0.5737,4027.03


In [ ]:
best_model = TransformerNextEvent(
    vocab_size=len(vocab),
    d_model=best_model_config["d_model"],
    nhead=best_model_config["nhead"],
    num_layers=best_model_config["num_layers"],
    dim_feedforward=best_model_config["dim_feedforward"],
    dropout=best_model_config["dropout"],
    max_len=best_model_config["max_len"],
    pad_id=vocab["<PAD>"]
).to(device)

best_model.load_state_dict(best_model_state)
best_model.eval()

TransformerNextEvent(
  (embedding): Embedding(18, 128, padding_idx=0)
  (pos_encoding): PositionalEncoding()
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear(in_features=128, out_features=256, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=256, out_features=128, bias=True)
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (dropout): Dropout(p=0.1, inplace=False)
  (fc): Linear(in_features=128, out_features=18, bias=True)
)

In [ ]:
print("Best run:", best_model_run_name)
print(best_model_config)

Best run: dropout_01
{'run_name': 'dropout_01', 'd_model': 128, 'nhead': 4, 'num_layers': 2, 'dim_feedforward': 256, 'dropout': 0.1, 'lr': 0.0003, 'weight_decay': 0.0001, 'max_len': 28, 'num_epochs': 7, 'patience': 3, 'max_grad_norm': 1.0}


In [ ]:
aggregation_methods = ["mean", "max", "topk_mean", "p95"]
agg_results = []

for agg in aggregation_methods:
    print(f"\nEvaluating aggregation: {agg}")



    result = evaluate_transformer_detector(
        model=best_model,
        val_df=val_df,
        test_df=test_df,
        vocab=vocab,
        max_len=best_model_config["max_len"],
        device=device,
        agg=agg,
        top_k=3
    )

    agg_results.append({
        "agg": agg,
        "val_f1": result["val_f1"],
        "test_f1": result["f1"],
        "precision": result.get("precision"),
        "recall": result.get("recall"),
        "roc_auc": result.get("roc_auc"),
        "pr_auc": result.get("pr_auc"),
        "threshold": result.get("threshold")
    })

agg_results_df = (
    pd.DataFrame(agg_results)
    .sort_values(by="val_f1", ascending=False)
    .reset_index(drop=True)
)

best_agg = agg_results_df.loc[0, "agg"]

print("\nBest aggregation:", best_agg)
display(agg_results_df)


Evaluating aggregation: mean

Evaluating aggregation: max

Evaluating aggregation: topk_mean

Evaluating aggregation: p95

Best aggregation: max


,agg,val_f1,test_f1,precision,recall,roc_auc,pr_auc,threshold
0,max,0.938741,0.871767,0.868492,0.875068,0.998286,0.879605,8.880089
1,topk_mean,0.824344,0.862967,0.790373,0.950243,0.999004,0.913929,3.854090
2,mean,0.688016,0.729341,0.577982,0.988102,0.997634,0.824769,0.627036
3,p95,0.583721,0.718119,0.926496,0.586263,0.978573,0.781301,5.283834


In [ ]:
final_result = evaluate_transformer_detector(
    model=best_model,
    val_df=val_df,
    test_df=test_df,
    vocab=vocab,
    max_len=best_model_config["max_len"],
    device=device,
    agg=best_agg,
    top_k=3
)

print("Final best run:", best_model_run_name)
print("Final best config:", best_model_config)
print("Final val_f1:", final_result["val_f1"])
print("Final test_f1:", final_result["f1"])
print("Final precision:", final_result.get("precision"))
print("Final recall:", final_result.get("recall"))
print("Final pr_auc:", final_result.get("pr_auc"))
print("Final roc_auc:", final_result.get("roc_auc"))
print("Final threshold:", final_result["threshold"])

Final best run: dropout_01
Final best config: {'run_name': 'dropout_01', 'd_model': 128, 'nhead': 4, 'num_layers': 2, 'dim_feedforward': 256, 'dropout': 0.1, 'lr': 0.0003, 'weight_decay': 0.0001, 'max_len': 28, 'num_epochs': 7, 'patience': 3, 'max_grad_norm': 1.0}
Final val_f1: 0.9387412587412587
Final test_f1: 0.8717672413793104
Final precision: 0.868491680085883
Final recall: 0.8750676041103299
Final pr_auc: 0.8796053065230285
Final roc_auc: 0.9982856217742556
Final threshold: 8.880088806152344
